In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)

# Keep only the two statuses being compared
status_df = df[
    df["status"].isin(["Ready to move", "Under Construction"])
].copy()

# Overall status comparison
status_summary = (
    status_df.groupby("status")
    .agg(
        listings=("status", "size"),
        avg_price_lakh=("price_lakh", "mean"),
        median_price_lakh=("price_lakh", "median"),
        avg_rate_per_sqft=("rate_per_sqft", "mean"),
        median_rate_per_sqft=("rate_per_sqft", "median"),
        avg_area_sqft=("area_sqft", "mean"),
        median_area_sqft=("area_sqft", "median")
    )
    .reset_index()
)

print("Ready-to-move vs under-construction:")
print(status_summary.to_string(index=False))


# Property type distribution
status_property_type = pd.crosstab(
    status_df["status"],
    status_df["flat_type"]
)

print("\nProperty type by status:")
print(status_property_type.to_string())


# Property type share within each status
status_property_type_pct = pd.crosstab(
    status_df["status"],
    status_df["flat_type"],
    normalize="index"
).mul(100).round(2)

print("\nProperty type share within each status (%):")
print(status_property_type_pct.to_string())


# BHK distribution
status_bhk = pd.crosstab(
    status_df["status"],
    status_df["bhk_group"]
)

print("\nBHK by status:")
print(status_bhk.to_string())


# RERA approval distribution
status_rera = pd.crosstab(
    status_df["status"],
    status_df["rera_approval"]
)

print("\nRERA approval by status:")
print(status_rera.to_string())


# RERA approval share within each status
status_rera_pct = pd.crosstab(
    status_df["status"],
    status_df["rera_approval"],
    normalize="index"
).mul(100).round(2)

print("\nRERA approval share within each status (%):")
print(status_rera_pct.to_string())


# Median comparison
ready = status_summary[
    status_summary["status"] == "Ready to move"
].iloc[0]

under_construction = status_summary[
    status_summary["status"] == "Under Construction"
].iloc[0]

median_price_difference = (
    ready["median_price_lakh"] -
    under_construction["median_price_lakh"]
)

median_rate_difference = (
    ready["median_rate_per_sqft"] -
    under_construction["median_rate_per_sqft"]
)

median_area_difference = (
    ready["median_area_sqft"] -
    under_construction["median_area_sqft"]
)

print("\nObserved median differences:")
print(f"Price difference (₹ lakh): {median_price_difference:.2f}")
print(f"Rate difference (₹/sqft): {median_rate_difference:.2f}")
print(f"Area difference (sqft): {median_area_difference:.0f}")

Ready-to-move vs under-construction:
            status  listings  avg_price_lakh  median_price_lakh  avg_rate_per_sqft  median_rate_per_sqft  avg_area_sqft  median_area_sqft
     Ready to move      7825      415.151144              260.0       13549.225304               11646.0    2695.617252            2225.0
Under Construction      5269      395.794564              286.0       17802.449042               14750.0    2111.511292            1855.0

Property type by status:
flat_type           Apartment  Floor  House  Penthouse  Plot  Villa
status                                                             
Ready to move            5922    717     31         31   921    203
Under Construction       4973    164     14          1   115      2

Property type share within each status (%):
flat_type           Apartment  Floor  House  Penthouse   Plot  Villa
status                                                              
Ready to move           75.68   9.16   0.40       0.40  11.77   2.59